# 7-1절 연습 문제 풀이

이 노트북은 7-1절 연습 문제(7-1 ~ 7-3)의 풀이 예시다. 세 문제 모두 코드보다 설명이 중심이지만,
주장을 숫자로 뒷받침할 수 있는 부분은 직접 계산해 확인한다.

- 본문 예제 코드는 `code_examples/ch07/07-01_example.ipynb`를 참고한다.
- 각 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)
print(f'파이토치 버전: {torch.__version__}')

파이토치 버전: 2.7.0+cu126


## 연습 문제 7-1

> 자연어를 처리하는 모델을 설계 중이다. 단어를 토큰으로 했을 때 어휘 사전을 구성하는 고유 토큰의 수가
> 1만 개라고 가정하자. 이 토큰을 원-핫 인코딩으로 표현하는 방식과 임베딩 차원이 128인 임베딩 벡터로
> 표현하는 방식을 다음 두 관점에서 비교해 보자.
> - 메모리 및 연산 효율성
> - 단어 간 관계의 표현 능력

In [2]:
VOCAB_SIZE = 10_000
EMBEDDING_DIM = 128
BATCH_SIZE = 64
SEQUENCE_LENGTH = 20
BYTES_PER_FLOAT = 4        # float32 기준

# (1) 배치 하나를 표현하는 데 필요한 메모리
onehot_batch = BATCH_SIZE * SEQUENCE_LENGTH * VOCAB_SIZE * BYTES_PER_FLOAT
embedding_batch = BATCH_SIZE * SEQUENCE_LENGTH * EMBEDDING_DIM * BYTES_PER_FLOAT

print(f'배치 하나의 입력 텐서 크기 (배치 {BATCH_SIZE}, 길이 {SEQUENCE_LENGTH})')
print(f'  원-핫 인코딩: ({BATCH_SIZE}, {SEQUENCE_LENGTH}, {VOCAB_SIZE:,}) = {onehot_batch / 1024**2:8.2f} MB')
print(f'  임베딩 벡터 : ({BATCH_SIZE}, {SEQUENCE_LENGTH}, {EMBEDDING_DIM}) = {embedding_batch / 1024**2:8.2f} MB')
print(f'  비율        : {onehot_batch / embedding_batch:.1f}배')

배치 하나의 입력 텐서 크기 (배치 64, 길이 20)
  원-핫 인코딩: (64, 20, 10,000) =    48.83 MB
  임베딩 벡터 : (64, 20, 128) =     0.62 MB
  비율        : 78.1배


In [3]:
# (2) 뒤따르는 LSTM 계층의 파라미터 수 비교
#     LSTM 파라미터 = 4 * (input_size * hidden + hidden * hidden + 2 * hidden)
HIDDEN_SIZE = 128

def lstm_parameters(input_size, hidden_size):
    return 4 * (input_size * hidden_size + hidden_size * hidden_size + 2 * hidden_size)

onehot_lstm = lstm_parameters(VOCAB_SIZE, HIDDEN_SIZE)
embedding_table = VOCAB_SIZE * EMBEDDING_DIM          # 임베딩 행렬 자체의 파라미터
embedding_lstm = lstm_parameters(EMBEDDING_DIM, HIDDEN_SIZE)

print(f'{"구성":>28} {"파라미터 수":>14}')
print('-' * 46)
print(f'{"원-핫 -> LSTM":>28} {onehot_lstm:14,d}')
print(f'{"임베딩 행렬":>28} {embedding_table:14,d}')
print(f'{"임베딩 -> LSTM":>28} {embedding_lstm:14,d}')
print(f'{"임베딩 방식 합계":>28} {embedding_table + embedding_lstm:14,d}')
print()
print('실제로 파이토치 계층을 만들어 확인')
print(f'  nn.LSTM({VOCAB_SIZE}, {HIDDEN_SIZE}): '
      f'{sum(p.numel() for p in nn.LSTM(VOCAB_SIZE, HIDDEN_SIZE, batch_first=True).parameters()):,}')
print(f'  nn.Embedding({VOCAB_SIZE}, {EMBEDDING_DIM}): '
      f'{sum(p.numel() for p in nn.Embedding(VOCAB_SIZE, EMBEDDING_DIM).parameters()):,}')
print(f'  nn.LSTM({EMBEDDING_DIM}, {HIDDEN_SIZE}): '
      f'{sum(p.numel() for p in nn.LSTM(EMBEDDING_DIM, HIDDEN_SIZE, batch_first=True).parameters()):,}')

                          구성         파라미터 수
----------------------------------------------
                 원-핫 -> LSTM      5,186,560
                      임베딩 행렬      1,280,000
                 임베딩 -> LSTM        132,096
                   임베딩 방식 합계      1,412,096

실제로 파이토치 계층을 만들어 확인
  nn.LSTM(10000, 128): 5,186,560
  nn.Embedding(10000, 128): 1,280,000
  nn.LSTM(128, 128): 132,096


In [4]:
# (3) 임베딩 벡터가 관계를 표현할 수 있는지 '표현 가능한 공간'의 관점에서 확인
#     원-핫 벡터끼리의 거리와 각도는 어떤 두 토큰을 골라도 항상 같다
onehot = torch.eye(5)          # 토큰 5개의 원-핫 벡터
print('원-핫 벡터 사이의 거리와 코사인 유사도 (토큰 0 기준)')
for i in range(1, 5):
    distance = torch.dist(onehot[0], onehot[i]).item()
    cosine = torch.nn.functional.cosine_similarity(onehot[0], onehot[i], dim=0).item()
    print(f'  토큰 0 - 토큰 {i}: 거리 {distance:.4f}, 코사인 유사도 {cosine:.4f}')

print()
print('임베딩 벡터는 값이 실수라 거리와 각도가 토큰마다 다를 수 있다 (무작위 초기화 예시)')
embedding = nn.Embedding(5, 4)
vectors = embedding.weight.detach()
for i in range(1, 5):
    distance = torch.dist(vectors[0], vectors[i]).item()
    cosine = torch.nn.functional.cosine_similarity(vectors[0], vectors[i], dim=0).item()
    print(f'  토큰 0 - 토큰 {i}: 거리 {distance:.4f}, 코사인 유사도 {cosine:+.4f}')

원-핫 벡터 사이의 거리와 코사인 유사도 (토큰 0 기준)
  토큰 0 - 토큰 1: 거리 1.4142, 코사인 유사도 0.0000
  토큰 0 - 토큰 2: 거리 1.4142, 코사인 유사도 0.0000
  토큰 0 - 토큰 3: 거리 1.4142, 코사인 유사도 0.0000
  토큰 0 - 토큰 4: 거리 1.4142, 코사인 유사도 0.0000

임베딩 벡터는 값이 실수라 거리와 각도가 토큰마다 다를 수 있다 (무작위 초기화 예시)
  토큰 0 - 토큰 1: 거리 2.0828, 코사인 유사도 -0.0956
  토큰 0 - 토큰 2: 거리 1.5393, 코사인 유사도 +0.0018
  토큰 0 - 토큰 3: 거리 1.0169, 코사인 유사도 +0.7068
  토큰 0 - 토큰 4: 거리 1.9402, 코사인 유사도 +0.6958


### 풀이 해설

**관점 1: 메모리 및 연산 효율성**

숫자로 보면 차이가 분명하다.

**입력 텐서의 크기**: 배치 64, 길이 20인 배치 하나를 표현하는 데
원-핫은 `(64, 20, 10000)`으로 약 **48.8MB**, 임베딩은 `(64, 20, 128)`으로 약 **0.6MB**다.
**78배 차이**이고, 이는 곧 어휘 사전 크기 ÷ 임베딩 차원(10000 ÷ 128 ≈ 78)이다.

**뒤따르는 계층의 파라미터 수**: 이쪽이 더 결정적이다.
원-핫 벡터를 `hidden_size=128`인 LSTM에 바로 넣으면 `input_size`가 10,000이 되어
**LSTM 파라미터만 5,186,560개**가 된다. 임베딩을 거치면 `input_size`가 128이므로
**LSTM 파라미터가 132,096개**로 **39분의 1**이 된다.
임베딩 행렬(10,000 × 128 = 1,280,000개)을 더해도 합계가 1,412,096개로,
**원-핫 방식의 약 27%(4분의 1이 조금 넘는 수준)**다.

게다가 임베딩 행렬은 **인덱싱으로 읽을 뿐 행렬곱을 하지 않는다**(본문 p5).
원-핫 방식은 매 토큰마다 1×10,000 벡터와 곱셈을 하는데 그중 9,999개가 0과의 곱이다. 계산의 99.99%가 낭비다.

**관점 2: 단어 간 관계의 표현 능력**

이쪽은 '효율의 차이'가 아니라 **'가능한가 불가능한가'의 차이**다.

원-핫 벡터는 **어떤 두 토큰을 골라도 거리가 √2로 같고 코사인 유사도가 0으로 같다**(위 실행 결과).
즉 모든 토큰이 서로 똑같이 멀다. '개'와 '강아지'가 '개'와 '컴퓨터'보다 가깝다는 사실을
**표현할 방법 자체가 없다.**

반면 임베딩 벡터는 요소가 실수라 토큰마다 거리와 각도가 다를 수 있다.
학습을 거치면 이 거리와 각도에 의미가 담긴다. 본문 p4가 말한 대로
'(소 벡터 − 개 벡터)와 (송아지 벡터 − 강아지 벡터)가 비슷하다'는 식의 **관계까지 방향으로 표현**된다.
본문 p8의 GloVe 가족 관계 시각화가 그 증거다.

**정리하면**, 메모리와 연산은 **정도의 차이**(78배, 3배)이지만 관계 표현은 **종류의 차이**다.
원-핫은 아무리 차원을 키워도 관계를 담을 수 없고, 임베딩은 훨씬 작은 차원으로도 담을 수 있다.
본문 p2가 원-핫의 문제를 '비효율성'과 '의미 없음' 둘로 나눈 이유가 여기에 있다.

### 문제 검토

- **적절성: 적합. 7장의 첫 연습 문제로 알맞다.** 본문 p2가 원-핫 인코딩의 문제를 '비효율성'과 '의미 없음' 둘로
  나누어 설명했는데, 이 문제의 두 관점이 그것과 정확히 대응한다. 본문을 제대로 읽었는지 확인하는 물음이다.
- **[검토] 어휘 사전 크기 1만, 임베딩 차원 128이라는 구체적인 수치를 준 것이 좋다.**
  수치가 있어야 '78배'처럼 손에 잡히는 답이 나온다. 수치 없이 '비교해 보자'라고만 했다면
  '임베딩이 더 효율적이다' 수준에서 끝났을 것이다.
- **★ [검토] 그런데 '계산해 보자'는 요구가 없어 실제로 세어 보지 않게 된다.**
  이 문제의 가치는 **10000 ÷ 128 ≈ 78이라는 숫자를 직접 얻는 것**에 있다.
  특히 **뒤따르는 LSTM 계층의 파라미터 수**까지 세어 보면, 임베딩의 이득이 입력 텐서 크기뿐 아니라
  모델 전체 크기에서 온다는 것이 드러난다(512만 개 → 13만 개).
  본문 p2는 '벡터가 길어진다'까지만 말하고 이 부분은 다루지 않으므로, 연습 문제에서 짚으면 값어치가 크다.
- **[검토] 두 번째 관점의 답이 첫 번째와 성격이 다르다는 점도 좋다.** 효율성은 정도의 차이인데
  관계 표현은 '가능/불가능'의 차이다. 두 관점을 나란히 묻는 구성이 이 대비를 드러낸다.

**윤문안**

> **7-1** 자연어를 처리하는 모델을 설계 중이다. 단어를 토큰으로 했을 때 어휘 사전을 구성하는 고유 토큰의 수가
> 1만 개라고 가정하자. 이 토큰을 원-핫 인코딩으로 표현하는 방식과 임베딩 차원이 128인 임베딩 벡터로
> 표현하는 방식을 다음 두 관점에서 비교해 보자.
> - 메모리 및 연산 효율성. 배치 하나의 입력 텐서 크기와, 뒤따르는 LSTM 계층의 파라미터 수를 직접 계산해 비교해 보자.
> - 단어 간 관계의 표현 능력

## 연습 문제 7-2

> [연습 문제 7-1]의 자연어 모델의 목적을 하나 설정하고, 그 목적을 기준으로 임베딩 행렬이 만들어지는 과정을
> 설명해 보자. 그리고 그 결과로 임베딩 행렬의 가중치가 어떤 특성을 갖게 될지 이야기해 보자.

### 목적 설정 — 영화 리뷰 감성 분류

1만 개의 단어로 이루어진 영화 리뷰를 읽고 **긍정인지 부정인지 분류하는 모델**을 목적으로 잡는다.
구조는 본문 [그림 7-4]와 같다.

```
입력(토큰 고유 번호) -> 임베딩 계층 -> LSTM 계층 -> 완전 연결 계층 -> 긍정/부정 로짓
```

### 풀이 해설

**임베딩 행렬이 만들어지는 과정**

본문 p5가 말한 대로, 임베딩 행렬은 **편향도 활성화 함수도 없는 특수한 선형 계층의 가중치**다.
따라서 다른 계층과 똑같은 방식으로 학습된다. 네 단계로 나누어 보면 이렇다.

**1단계 — 무작위 초기화.** `nn.Embedding(10000, 128)`을 만들면 10,000 × 128 행렬이 무작위 값으로 채워진다.
이 시점에서 '좋다'와 '훌륭하다'의 벡터는 아무 관계가 없다. 어느 두 단어든 서로 남남이다.

**2단계 — 순전파.** 리뷰 하나가 들어오면 각 단어의 고유 번호로 임베딩 행렬의 **해당 행만 인덱싱**해 꺼낸다.
꺼낸 벡터들이 LSTM을 거쳐 마지막 숨겨진 상태가 되고, 완전 연결 계층이 긍정/부정 로짓을 낸다.

**3단계 — 손실과 역전파.** 예측이 틀리면 교차 엔트로피 손실이 커진다.
기울기가 완전 연결 계층 → LSTM → **임베딩 계층**의 순서로 거슬러 올라간다.

**4단계 — 갱신.** 여기가 핵심이다. **이번 배치에 등장한 단어의 행만 갱신된다.**
인덱싱으로 꺼냈으므로 나머지 9,900여 개 행에는 기울기가 흐르지 않는다.

이 과정을 수없이 반복하면, **'긍정/부정을 맞히는 데 도움이 되는 방향'으로 각 단어의 벡터가 조금씩 옮겨 간다.**
사람이 '좋다와 훌륭하다는 비슷하다'고 알려 준 적이 없는데도 말이다.

**결과로 가중치가 갖게 되는 특성**

**(1) 목적에 맞는 단어끼리 모인다.** 'good', 'great', 'excellent', 'wonderful'이 한쪽에,
'bad', 'terrible', 'awful'이 반대쪽에 모인다. 이들은 리뷰에서 같은 역할을 하므로
**비슷한 벡터를 가질 때 손실이 가장 낮아지기 때문**이다.

**(2) 목적과 무관한 관계는 담기지 않는다.** 이 점이 중요하다.
감성 분류 모델의 임베딩에서 'cat'과 'dog'가 가까워질 이유는 없다. 둘 다 감성에 중립적이므로
**구분할 필요가 없기 때문**이다. 본문 p4가 "만약 성체와 새끼의 차이가 더 중요한 모델이라면 …"이라고 한 대로,
**임베딩은 모델의 목적이 무엇이냐에 따라 전혀 다르게 만들어진다.**

**(3) 드물게 등장하는 단어의 벡터는 잘 학습되지 않는다.** 갱신은 등장한 단어에만 일어나므로,
전체 리뷰에서 다섯 번 나온 단어는 다섯 번밖에 갱신되지 않아 **거의 무작위 초기값에 머문다.**
실무에서 드문 토큰을 `<unk>`로 묶는 이유 중 하나다.

**(4) 각 축은 사람의 말로 설명되지 않는다.** 128개 축 중 어느 하나가 '긍정도'를 나타내지는 않는다.
본문 p5가 말한 **잠재 차원**이며, 의미는 축이 아니라 **거리와 방향**으로 드러난다.

**이 문제가 말하려는 것**은 결국 하나다. **임베딩은 목적의 부산물이다.**
'좋은 임베딩을 만들자'고 따로 학습하는 것이 아니라, 주어진 문제를 풀다 보니 만들어진다.

### 문제 검토

- **적절성: 적합. 7-1보다 한 단계 깊이 들어간다.** 7-1이 '임베딩이 왜 좋은가'를 묻는다면
  7-2는 '**어떻게 그렇게 되는가**'를 묻는다. 본문 p5의 "임베딩 행렬을 모델 안에서 함께 학습시키는 것만으로"라는
  한 문장을 독자가 자기 말로 풀어낼 수 있는지 확인한다.
- **★ [검토] '목적을 하나 설정하고'가 이 문제의 핵심이자 가장 좋은 부분이다.**
  목적을 정하지 않으면 '학습으로 만들어진다'는 일반론에서 끝난다. 목적이 정해져야
  **'그 목적에 유리한 방향으로만 정리된다'**는 결론에 이를 수 있고, 이것이 본문 p4가 강조한 내용이다.
- **[검토] 다만 '목적'의 범위가 넓어 답의 수준이 크게 갈릴 수 있다.** '자연어 처리 모델'이라는 조건만 있으므로,
  독자가 '번역'처럼 복잡한 목적을 고르면 임베딩이 어떻게 정리될지 설명하기 어려워진다.
  **예시를 하나 들어 주면** 난도가 안정된다(감성 분류, 다음 단어 예측 등).
- **[검토] '가중치가 어떤 특성을 갖게 될지'라는 물음이 좋다.** 그냥 '결과를 설명하라'가 아니라
  **가중치**라는 구체적인 대상을 지목해, 임베딩 행렬이 특별한 무엇이 아니라 **보통의 가중치 텐서**라는
  본문의 관점을 유지하게 한다.

**윤문안**

> **7-2** [연습 문제 7-1]의 자연어 모델의 목적을 하나 설정하고(예: 영화 리뷰의 긍정·부정 분류, 다음 단어 예측 등),
> 그 목적을 기준으로 임베딩 행렬이 만들어지는 과정을 설명해 보자. 그리고 그 결과로 임베딩 행렬의 가중치가
> 어떤 특성을 갖게 될지 이야기해 보자. 목적이 달라지면 같은 단어의 임베딩 벡터도 달라질지 함께 생각해 보자.

## 연습 문제 7-3

> 텍스트 데이터의 임베딩은 보통 글자를 토큰으로 사용하지 않고 단어 또는 어절(한국어 텍스트의 경우)을
> 토큰으로 사용한다. 그 이유가 무엇일까? 그리고 어떤 상황에서 평소와 달리 글자를 토큰으로 사용할까?

In [5]:
# 같은 텍스트를 글자 단위와 단어 단위로 토큰화해 비교한다
import re

SAMPLE = ('the scarecrow and the tin woodman walked along the yellow brick road '
          'toward the emerald city, and dorothy followed them with toto.')

char_tokens = list(SAMPLE)
word_tokens = re.findall(r"[a-z']+|[,.]", SAMPLE)

print(f'{"토큰 단위":>10} {"전체 토큰":>10} {"고유 토큰":>10} {"토큰 하나의 평균 정보량":>22}')
print('-' * 58)
for name, tokens in [('글자', char_tokens), ('단어', word_tokens)]:
    print(f'{name:>10} {len(tokens):10,d} {len(set(tokens)):10,d} '
          f'{len(SAMPLE) / len(tokens):18.2f}자')

print()
# 같은 의미를 담으려면 윈도우가 얼마나 길어야 하는지 확인
print("'the yellow brick road'를 윈도우 안에 담으려면")
print(f"  단어 단위: {len(re.findall(chr(114)+chr(39), '')) or 4}개 토큰")
print(f"  글자 단위: {len('the yellow brick road')}개 토큰")

     토큰 단위      전체 토큰      고유 토큰          토큰 하나의 평균 정보량
----------------------------------------------------------
        글자        130         22               1.00자
        단어         24         20               5.42자

'the yellow brick road'를 윈도우 안에 담으려면
  단어 단위: 4개 토큰
  글자 단위: 21개 토큰


### 풀이 해설

**왜 보통 단어(또는 어절)를 쓰는가**

이유는 세 가지다.

**1. 토큰 하나가 담는 의미의 크기.** 이것이 가장 근본적인 이유다.
임베딩은 **토큰 하나에 하나의 벡터를 배정**한다. 그런데 글자 `a`에 무슨 의미를 담을 수 있을까?
`a`는 `cat`에도 `apple`에도 `walked`에도 들어가는데, 이 모든 맥락을 벡터 하나로 표현할 수는 없다.
반면 `apple`이라는 단어에는 '과일', '둥글다', '먹는다' 같은 의미를 담을 수 있다.
**의미의 최소 단위가 단어이기 때문에, 의미를 담는 임베딩도 단어 단위가 자연스럽다.**

**2. 문맥을 담는 데 필요한 윈도우의 길이.** 위 실행 결과에서 보듯 같은 문장이 글자 단위로는 훨씬 많은 토큰이 된다.
`the yellow brick road`라는 구절을 윈도우 안에 담으려면 단어 단위로는 4개면 되지만
**글자 단위로는 21개**가 필요하다. 순환 신경망은 시점마다 한 번씩 계산하므로,
**같은 문맥을 보려고 다섯 배 넘게 계산해야 한다.** 게다가 시점이 길어질수록 장기 의존성 문제도 심해진다(6장).

**3. 앞의 두 이유가 합쳐진 결과 — 학습이 쉬워진다.** 글자 단위 모델은 '단어를 이루는 철자 규칙'부터
'단어의 의미'까지 전부 스스로 배워야 한다. 단어 단위 모델은 **철자 단계를 건너뛰고 의미에서 시작**한다.

**그렇다면 왜 글자를 쓰는 경우가 있는가**

단어 단위의 약점이 치명적인 상황들이다.

**(1) 어휘 사전이 감당할 수 없이 커질 때.** 한국어가 대표적이다.
'강변', '강변은', '강변에서', '강변까지'가 모두 다른 어절이 되어 어휘 사전이 폭발한다.
글자는 아무리 긴 글이라도 수천 개에서 멈춘다.

**(2) 처음 보는 단어를 다뤄야 할 때.** 단어 단위 모델은 학습에 없던 단어를 만나면 `<unk>`로 뭉갤 수밖에 없다.
신조어, 고유명사, 오타가 많은 텍스트(SNS, 검색 질의)에서는 이것이 큰 손실이다.
글자 단위는 **어떤 단어든 글자로 쪼개 표현할 수 있다.**

**(3) 문제 자체가 글자 수준일 때.** **6-3절의 띄어쓰기 모델이 바로 이 경우다.**
'어디에서 단어가 끝나는가'를 판단하는 문제에서 단어 단위 토큰을 쓰는 것은 앞뒤가 맞지 않는다.
이미 단어를 나눠 놓고 단어 경계를 찾는 셈이기 때문이다.
철자 교정, 형태소 분석, 고유명사 인식 같은 문제도 비슷하다.

**(4) 데이터가 적을 때.** 어휘 사전이 크면 단어 하나당 등장 횟수가 줄어 학습이 어렵다.
글자 단위는 어휘 사전이 작아 각 토큰이 충분히 자주 등장한다.

**덧붙임 — 오늘날의 실제 해법**

두 방식의 절충으로 **서브워드**(subword) 단위가 널리 쓰인다.
`unhappiness`를 `un` + `happi` + `ness`처럼 쪼개면, 어휘 사전은 적당한 크기로 유지하면서
처음 보는 단어도 표현할 수 있고 의미도 어느 정도 담긴다.
GPT나 BERT 같은 모델이 모두 이 방식을 쓴다. 10장 이후에 다시 만나게 된다.

### 문제 검토

- **적절성: 적합. 특히 두 번째 물음이 이 문제를 살린다.** 첫 번째 물음만 있었다면 '단어가 의미 단위니까'로
  끝났을 텐데, "**어떤 상황에서 평소와 달리 글자를 토큰으로 사용할까**"가 붙어 있어
  **규칙이 아니라 판단**의 문제임을 알게 한다. 좋은 설계다.
- **★ [검토] 두 번째 물음의 답이 이미 이 책 안에 있다는 점을 활용하면 좋겠다.**
  **6-3절의 띄어쓰기 모델이 정확히 '글자를 토큰으로 쓴 사례'**다. 그것도 이유가 분명하다.
  단어 경계를 찾는 문제에서 단어 토큰을 쓸 수는 없기 때문이다.
  독자가 바로 앞 장에서 만든 모델을 떠올리면 답이 훨씬 구체적이 되는데,
  지금 지문은 이 연결을 열어 두지 않아 일반론에 머물기 쉽다.
- **[검토] '어절(한국어 텍스트의 경우)'이라고 괄호로 밝힌 것이 정확하다.** 영어의 word와 한국어의 어절이
  같지 않다는 점을 짚어 준다. 다만 **한국어에서 어절 단위가 특히 불리한 이유**(조사·어미로 어휘가 폭발)는
  두 번째 물음의 답에서 다뤄야 할 내용이라 여기서는 말하지 않는 편이 낫다.
- **[검토] 서브워드를 언급할지 판단이 필요하다.** 오늘날의 실제 해법이지만 이 책의 범위를 넘고,
  10장 이후에 다시 다룰 수 있으므로 **연습 문제에서는 언급하지 않는 현재 상태가 적절하다.**

**윤문안**

> **7-3** 텍스트 데이터의 임베딩은 보통 글자를 토큰으로 사용하지 않고 단어 또는 어절(한국어 텍스트의 경우)을
> 토큰으로 사용한다. 그 이유가 무엇일까? 그리고 어떤 상황에서 평소와 달리 글자를 토큰으로 사용할까?
> 6장에서 만든 두 모델이 각각 어떤 단위를 토큰으로 사용했는지, 그리고 그 선택이 왜 그 문제에 알맞았는지
> 떠올려 보면 도움이 된다.